# DroneRF XAI Notebook: VGG16-BN, Swin-Tiny, ViT-Small

This notebook follows the model-loading pattern used in your test notebooks and runs:
- Grad-CAM (with transformer-safe token handling)
- LIME
- SHAP

Then it reports **Fidelity**, **Stability**, and **Sparsity** for each method and model.

In [ ]:
# If running on Kaggle/clean env, uncomment installs:
# !pip install -q timm lime shap scikit-image

import os
import math
import re
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from PIL import Image

import timm
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr

from lime import lime_image
from skimage.segmentation import slic
import shap

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# -----------------------------
# Config (edit these for your dataset/checkpoints)
# -----------------------------
DATA_ROOTS = [
    '/kaggle/input/datasets/samir5488377/new-noisy-spectrograms/images/snr_clean',
    '/kaggle/input/datasets/samir5488377/new-noisy-spectrograms/images/snr_20',
    '/kaggle/input/datasets/samir5488377/new-noisy-spectrograms/images/snr_15',
    '/kaggle/input/datasets/samir5488377/new-noisy-spectrograms/images/snr_10',
    '/kaggle/input/datasets/samir5488377/new-noisy-spectrograms/images/snr_5',
    '/kaggle/input/datasets/samir5488377/new-noisy-spectrograms/images/snr_0',
]
WEIGHT_ROOT = '/kaggle/input/datasets/devil1696/dronerf-split'

CLASS_NAMES = sorted(['ardrone', 'background', 'bepop', 'phantom'])
NUM_CLASSES = len(CLASS_NAMES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_NAMES)}

IMG_SIZE = 224
BATCH_SIZE = 64
TEST_SPLIT = 0.15
VAL_SPLIT = 0.15

MODEL_NAMES = ['vgg16_bn', 'swin_tiny', 'vit_small']
WEIGHT_FILES = {
    'vgg16_bn': 'best_vgg16_bn.pth',
    'swin_tiny': 'best_swin_tiny_patch4_window7_224.pth',
    'vit_small': 'best_vit_small_patch16_224.pth',
}

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

N_EVAL_SAMPLES = 12
PER_CLASS_PER_SNR = 1
MASK_TOP_FRACTION = 0.10
STABILITY_PERTURBATIONS = 3
NOISE_STD = 0.03
LIME_SAMPLES = 500
SHAP_MAX_EVALS = 300
SNR_PARSE_DEFAULT = 0

assert all(k in WEIGHT_FILES for k in MODEL_NAMES), 'Missing checkpoint entries in WEIGHT_FILES'
print('Classes:', CLASS_NAMES)
print('Models:', MODEL_NAMES)

In [ ]:
# -----------------------------
# Dataset loading (class-folder structure)
# -----------------------------
def parse_snr_from_path(path):
    p = Path(path)
    parts = [s.lower() for s in p.parts]
    for token in reversed(parts):
        if 'clean' in token:
            return 999
        m = re.search(r'snr[_]?([+-]?\d+)', token)
        if m:
            return int(m.group(1))
    stem = p.stem.lower()
    m2 = re.search(r'snr[_]?([+-]?\d+)', stem)
    if m2:
        return int(m2.group(1))
    return SNR_PARSE_DEFAULT

def load_dataset(data_roots):
    paths, labels, snrs = [], [], []
    for root in data_roots:
        root_path = Path(root)
        if not root_path.exists():
            continue
        for class_name in CLASS_NAMES:
            class_dir = root_path / class_name
            if not class_dir.exists():
                continue
            for p in sorted(class_dir.glob('*.png')):
                paths.append(str(p))
                labels.append(CLASS_TO_IDX[class_name])
                snrs.append(parse_snr_from_path(p))
    return paths, np.array(labels, dtype=np.int64), np.array(snrs, dtype=np.int32)

all_paths, all_labels, all_snrs = load_dataset(DATA_ROOTS)
print(f'Total images: {len(all_paths)}')
print(f'Parsed SNR levels: {sorted(set(all_snrs.tolist()))}')

idx = np.arange(len(all_paths))
train_idx, temp_idx = train_test_split(idx, test_size=VAL_SPLIT + TEST_SPLIT, random_state=SEED, stratify=all_labels)
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=TEST_SPLIT / (VAL_SPLIT + TEST_SPLIT),
    random_state=SEED,
    stratify=all_labels[temp_idx]
)

test_paths = [all_paths[i] for i in sorted(test_idx.tolist())]
test_labels = all_labels[sorted(test_idx.tolist())]
test_snrs = all_snrs[sorted(test_idx.tolist())]
print(f'Test images: {len(test_paths)}')
print(f'Test SNR levels: {sorted(set(test_snrs.tolist()))}')

def get_transform():
    return transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

transform = get_transform()

In [ ]:
# -----------------------------
# Model definitions (matching test notebooks)
# -----------------------------
class VGG16Model(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, pretrained=False):
        super().__init__()
        self.backbone = timm.create_model('vgg16_bn', pretrained=pretrained, num_classes=0)
        self.fc = nn.Sequential(
            nn.Linear(4096, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
        )
        self.head = nn.Linear(512, num_classes)

    def forward(self, x):
        return self.head(self.fc(self.backbone(x)))


class ViTSmallModel(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, pretrained=False, img_size=224):
        super().__init__()
        self.backbone = timm.create_model('vit_small_patch16_224', pretrained=pretrained, num_classes=0, img_size=img_size)
        self.head = nn.Linear(384, num_classes)

    def forward(self, x):
        return self.head(self.backbone(x))


class SwinTinyModel(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES, pretrained=False, img_size=224):
        super().__init__()
        self.backbone = timm.create_model('swin_tiny_patch4_window7_224', pretrained=pretrained, num_classes=0, img_size=img_size)
        self.head = nn.Linear(768, num_classes)

    def forward(self, x):
        return self.head(self.backbone(x))


def build_model(name):
    if name == 'vgg16_bn':
        return VGG16Model(num_classes=NUM_CLASSES, pretrained=False)
    if name == 'vit_small':
        return ViTSmallModel(num_classes=NUM_CLASSES, pretrained=False, img_size=IMG_SIZE)
    if name == 'swin_tiny':
        return SwinTinyModel(num_classes=NUM_CLASSES, pretrained=False, img_size=IMG_SIZE)
    raise ValueError(name)


def load_state_flexible(model, ckpt_path):
    raw = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    if isinstance(raw, dict) and 'state_dict' in raw:
        state = raw['state_dict']
    elif isinstance(raw, dict) and 'model_state_dict' in raw:
        state = raw['model_state_dict']
    else:
        state = raw
    if isinstance(state, dict):
        cleaned = {}
        for k, v in state.items():
            nk = k.replace('module.', '')
            nk = nk.replace('model.', '') if nk.startswith('model.') else nk
            cleaned[nk] = v
        state = cleaned
    missing, unexpected = model.load_state_dict(state, strict=False)
    return missing, unexpected


MODELS = {}
for mn in MODEL_NAMES:
    model = build_model(mn)
    ckpt = os.path.join(WEIGHT_ROOT, WEIGHT_FILES[mn])
    missing, unexpected = load_state_flexible(model, ckpt)
    model = model.to(device).eval()
    MODELS[mn] = model
    print(f'{mn}: loaded from {ckpt}')
    print(f'  missing={len(missing)}, unexpected={len(unexpected)}')

In [ ]:
# -----------------------------
# Shared prediction and image utilities
# -----------------------------
def preprocess_pil(pil_img):
    return transform(pil_img)

def np_to_pil(img_np):
    img_u8 = np.clip(img_np * 255.0, 0, 255).astype(np.uint8)
    return Image.fromarray(img_u8)

def load_img_np(path):
    img = Image.open(path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    return np.asarray(img, dtype=np.float32) / 255.0

def predict_logits(model_name, img_np_batch):
    model = MODELS[model_name]
    tensors = [preprocess_pil(np_to_pil(img_np)) for img_np in img_np_batch]
    x = torch.stack(tensors).to(device)
    with torch.no_grad():
        logits = model(x)
    return logits

def predict_proba(model_name, img_np_batch):
    logits = predict_logits(model_name, img_np_batch)
    return F.softmax(logits, dim=1).cpu().numpy()

def top_class_and_prob(model_name, img_np):
    probs = predict_proba(model_name, [img_np])[0]
    cls = int(np.argmax(probs))
    return cls, float(probs[cls])

def normalize_map(m):
    m = m.astype(np.float32)
    m = np.maximum(m, 0.0)
    mx = float(m.max())
    if mx > 0:
        m = m / mx
    return m

def overlay(img_np, attn, alpha=0.45):
    cmap = plt.get_cmap('jet')(attn)[..., :3]
    return np.clip((1 - alpha) * img_np + alpha * cmap, 0, 1)

In [ ]:
# -----------------------------
# Grad-CAM (transformer-safe)
# -----------------------------
GRADCAM_CANDIDATES = {
    'vgg16_bn': ['backbone.features.42'],
    'vit_small': ['backbone.norm'],
    'swin_tiny': ['backbone.norm'],
}

TRANSFORMER_GRID = {
    'vit_small': (14, 14),
    'swin_tiny': (7, 7),
}

def get_layer_by_name(model, layer_name):
    mod = model
    for p in layer_name.split('.'):
        mod = mod[int(p)] if p.isdigit() else getattr(mod, p)
    return mod

def resolve_gradcam_target(model, model_name):
    for candidate in GRADCAM_CANDIDATES[model_name]:
        try:
            _ = get_layer_by_name(model, candidate)
            return candidate
        except Exception:
            continue
    raise RuntimeError(f'No Grad-CAM target resolved for {model_name}')

GRADCAM_TARGET = {mn: resolve_gradcam_target(MODELS[mn], mn) for mn in MODEL_NAMES}
print('Grad-CAM targets:', GRADCAM_TARGET)

def disable_inplace_ops(model):
    originals = []
    for m in model.modules():
        if hasattr(m, 'inplace') and m.inplace:
            originals.append((m, True))
            m.inplace = False
    return originals

def restore_inplace_ops(originals):
    for m, val in originals:
        m.inplace = val

def gradcam_map(model_name, img_np, target_class=None):
    model = MODELS[model_name]
    model.eval()
    model.zero_grad()
    saved_inplace = disable_inplace_ops(model)

    x = preprocess_pil(np_to_pil(img_np)).unsqueeze(0).to(device).requires_grad_(True)
    layer = get_layer_by_name(model, GRADCAM_TARGET[model_name])

    activations = []
    gradients = []

    fh = layer.register_forward_hook(lambda m, i, o: activations.append(o.detach().clone()))
    bh = layer.register_full_backward_hook(lambda m, gi, go: gradients.append(go[0].detach().clone()))

    try:
        logits = model(x)
        probs = F.softmax(logits, dim=1)
        if target_class is None:
            target_class = int(torch.argmax(probs, dim=1).item())
        score = logits[0, target_class]
        score.backward()
    finally:
        fh.remove()
        bh.remove()
        restore_inplace_ops(saved_inplace)

    act = activations[0].detach()
    grad = gradients[0].detach()

    if act.dim() == 3:
        grid = TRANSFORMER_GRID[model_name]
        has_cls = (act.shape[1] == grid[0] * grid[1] + 1)
        if has_cls:
            act = act[:, 1:, :]
            grad = grad[:, 1:, :]
        act = act.permute(0, 2, 1).reshape(1, act.shape[2], grid[0], grid[1])
        grad = grad.permute(0, 2, 1).reshape(1, grad.shape[2], grid[0], grid[1])

    if act.dim() == 2:
        act = act.unsqueeze(-1).unsqueeze(-1)
        grad = grad.unsqueeze(-1).unsqueeze(-1)

    weights = grad.mean(dim=(2, 3), keepdim=True)
    cam = torch.relu((weights * act).sum(dim=1)).squeeze().detach().cpu().numpy()

    if cam.ndim == 0:
        cam = np.full((7, 7), float(cam), dtype=np.float32)
    elif cam.ndim == 1:
        side = int(math.sqrt(cam.shape[0]))
        cam = cam.reshape(side, side) if side * side == cam.shape[0] else cam.reshape(1, -1)

    cam = normalize_map(cam)
    cam = np.array(Image.fromarray((cam * 255).astype(np.uint8)).resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR), dtype=np.float32) / 255.0
    pred_prob = float(probs[0, target_class].item())
    return cam, target_class, pred_prob

In [ ]:
# -----------------------------
# LIME and SHAP maps
# -----------------------------
lime_explainer = lime_image.LimeImageExplainer(random_state=SEED)
shap_explainers = {}

def lime_map(model_name, img_np, target_class=None):
    probs = predict_proba(model_name, [img_np])[0]
    if target_class is None:
        target_class = int(np.argmax(probs))

    exp = lime_explainer.explain_instance(
        image=img_np,
        classifier_fn=lambda arr: predict_proba(model_name, arr),
        top_labels=1,
        num_samples=LIME_SAMPLES,
        segmentation_fn=lambda x: slic(x, n_segments=80, compactness=10, sigma=1),
    )

    segments = exp.segments
    heat = np.zeros_like(segments, dtype=np.float32)
    for seg_id, w in exp.local_exp[target_class]:
        heat[segments == seg_id] = w
    heat = normalize_map(np.maximum(heat, 0.0))
    return heat, target_class, float(probs[target_class])

def get_shap_explainer(model_name):
    if model_name not in shap_explainers:
        masker = shap.maskers.Image('blur(32,32)', (IMG_SIZE, IMG_SIZE, 3))
        shap_explainers[model_name] = shap.Explainer(
            lambda arr: predict_proba(model_name, arr),
            masker,
        )
    return shap_explainers[model_name]

def shap_map(model_name, img_np, target_class=None):
    probs = predict_proba(model_name, [img_np])[0]
    if target_class is None:
        target_class = int(np.argmax(probs))

    explainer = get_shap_explainer(model_name)
    sv = explainer(
        img_np[np.newaxis, ...],
        max_evals=SHAP_MAX_EVALS,
        outputs=[target_class],
    )

    vals = sv.values
    if vals.ndim == 5:
        # (B, H, W, C, O)
        heat = np.abs(vals[0, :, :, :, 0]).mean(axis=-1)
    elif vals.ndim == 4:
        # (B, H, W, C)
        heat = np.abs(vals[0]).mean(axis=-1)
    else:
        raise RuntimeError(f'Unexpected SHAP values shape: {vals.shape}')

    heat = normalize_map(heat)
    return heat, target_class, float(probs[target_class])

In [ ]:
# -----------------------------
# Fidelity, Stability, Sparsity (SNR-aware)
# -----------------------------
def mask_top_fraction(img_np, attn_map, frac=0.10):
    h = attn_map.reshape(-1)
    k = max(1, int(frac * h.size))
    idx = np.argpartition(h, -k)[-k:]
    mask = np.zeros(h.size, dtype=bool)
    mask[idx] = True
    mask = mask.reshape(attn_map.shape)

    out = img_np.copy()
    out[mask, :] = 0.0
    return out

def hoyer_sparsity(attn_map):
    x = np.abs(attn_map.reshape(-1).astype(np.float64))
    n = x.size
    if n <= 1:
        return 0.0
    l1 = np.sum(x)
    l2 = np.sqrt(np.sum(x * x)) + 1e-12
    return float((np.sqrt(n) - (l1 / l2)) / (np.sqrt(n) - 1.0))

def fidelity_drop(model_name, img_np, target_class, attn_map, frac=0.10):
    base_prob = predict_proba(model_name, [img_np])[0][target_class]
    masked = mask_top_fraction(img_np, attn_map, frac=frac)
    masked_prob = predict_proba(model_name, [masked])[0][target_class]
    return float(base_prob - masked_prob)

def stability_score(method_fn, model_name, img_np, target_class, n_pert=3, noise_std=0.03):
    base_map, _, _ = method_fn(model_name, img_np, target_class=target_class)
    base_vec = base_map.reshape(-1)
    corrs = []

    def safe_spearman(a, b, eps=1e-12):
        a_std = float(np.std(a))
        b_std = float(np.std(b))
        if a_std < eps and b_std < eps:
            return 1.0 if np.allclose(a, b) else 0.0
        if a_std < eps or b_std < eps:
            return 0.0
        c = spearmanr(a, b).correlation
        return 0.0 if np.isnan(c) else float(c)
    for _ in range(n_pert):
        noisy = np.clip(img_np + np.random.normal(0, noise_std, size=img_np.shape).astype(np.float32), 0, 1)
        pert_map, _, _ = method_fn(model_name, noisy, target_class=target_class)
        c = safe_spearman(base_vec, pert_map.reshape(-1))
        corrs.append(float(c))
    return float(np.mean(corrs))

METHODS = {
    'gradcam': gradcam_map,
    'lime': lime_map,
    'shap': shap_map,
}

def build_eval_indices_per_class_per_snr(labels, snrs, per_class_per_snr=1, max_total=12, seed=42):
    rng = np.random.default_rng(seed)
    labels = np.asarray(labels)
    snrs = np.asarray(snrs)
    picked = []
    unique_snrs = sorted(set(snrs.tolist()))
    for snr in unique_snrs:
        for cls in range(NUM_CLASSES):
            cand = np.where((labels == cls) & (snrs == snr))[0]
            if len(cand) == 0:
                continue
            take = min(per_class_per_snr, len(cand))
            chosen = rng.choice(cand, size=take, replace=False)
            picked.extend(chosen.tolist())
    picked = sorted(set(picked))
    if len(picked) > max_total:
        picked = rng.choice(np.array(picked), size=max_total, replace=False).tolist()
        picked = sorted(picked)
    return picked

eval_indices = build_eval_indices_per_class_per_snr(
    labels=test_labels,
    snrs=test_snrs,
    per_class_per_snr=PER_CLASS_PER_SNR,
    max_total=min(N_EVAL_SAMPLES, len(test_paths)),
    seed=SEED,
)
print(f'Evaluation samples selected: {len(eval_indices)}')

rows = []
for mn in MODEL_NAMES:
    print(f'\nEvaluating explanations for model: {mn}')
    for i in eval_indices:
        img_np = load_img_np(test_paths[int(i)])
        pred_cls, _ = top_class_and_prob(mn, img_np)
        true_cls = int(test_labels[int(i)])
        snr_db = int(test_snrs[int(i)])

        for method_name, method_fn in METHODS.items():
            try:
                attn, cls, _ = method_fn(mn, img_np, target_class=pred_cls)
                fid = fidelity_drop(mn, img_np, cls, attn, frac=MASK_TOP_FRACTION)
                stab = stability_score(
                    method_fn, mn, img_np, cls,
                    n_pert=STABILITY_PERTURBATIONS,
                    noise_std=NOISE_STD,
                )
                spar = hoyer_sparsity(attn)
                rows.append({
                    'model': mn,
                    'method': method_name,
                    'sample_index': int(i),
                    'snr_db': snr_db,
                    'true_class': true_cls,
                    'target_class': int(cls),
                    'fidelity_drop': fid,
                    'stability_spearman': stab,
                    'sparsity_hoyer': spar,
                })
            except Exception as ex:
                print(f'  [{method_name}] sample {int(i)} failed: {ex}')

metrics_df = pd.DataFrame(rows)
summary_df = metrics_df.groupby(['model', 'method'], as_index=False).agg({
    'fidelity_drop': ['mean', 'std'],
    'stability_spearman': ['mean', 'std'],
    'sparsity_hoyer': ['mean', 'std'],
})
summary_df.columns = [
    'model', 'method',
    'fidelity_mean', 'fidelity_std',
    'stability_mean', 'stability_std',
    'sparsity_mean', 'sparsity_std',
]

snr_summary_df = metrics_df.groupby(['model', 'method', 'snr_db'], as_index=False).agg({
    'fidelity_drop': 'mean',
    'stability_spearman': 'mean',
    'sparsity_hoyer': 'mean',
    'sample_index': 'count',
})
snr_summary_df = snr_summary_df.rename(columns={'sample_index': 'n_samples'})

metrics_df.to_csv('xai_metrics_per_sample.csv', index=False)
summary_df.to_csv('xai_metrics_summary.csv', index=False)
snr_summary_df.to_csv('xai_metrics_by_snr.csv', index=False)

print('Saved: xai_metrics_per_sample.csv')
print('Saved: xai_metrics_summary.csv')
print('Saved: xai_metrics_by_snr.csv')
display(summary_df.sort_values(['model', 'method']))

In [ ]:
# -----------------------------
# SNR-vs-Fidelity plot + one sample per SNR for the most interesting model
# -----------------------------
if len(snr_summary_df) == 0:
    raise RuntimeError('snr_summary_df is empty. Run the metrics cell first.')

# 1) SNR vs Fidelity plot for all models and methods
for method_name in ['gradcam', 'lime', 'shap']:
    fig, ax = plt.subplots(figsize=(8, 4))
    sub = snr_summary_df[snr_summary_df['method'] == method_name]
    for mn in MODEL_NAMES:
        cur = sub[sub['model'] == mn].sort_values('snr_db')
        if len(cur) == 0:
            continue
        ax.plot(cur['snr_db'].values, cur['fidelity_drop'].values, marker='o', linewidth=1.6, label=mn)
    ax.set_title(f'SNR vs Fidelity ({method_name.upper()})')
    ax.set_xlabel('SNR (dB)')
    ax.set_ylabel('Mean Fidelity Drop')
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

# 2) Most interesting model = highest mean fidelity across all methods/samples
model_interest = summary_df.groupby('model', as_index=False)['fidelity_mean'].mean().sort_values('fidelity_mean', ascending=False)
most_interesting_model = model_interest.iloc[0]['model']
print(f'Most interesting model by mean fidelity: {most_interesting_model}')

# 3) One sample per SNR level (for most interesting model)
snr_to_sample = {}
for s in sorted(set(test_snrs.tolist())):
    cand = [i for i in eval_indices if int(test_snrs[int(i)]) == int(s)]
    if len(cand) == 0:
        cand = np.where(test_snrs == s)[0].tolist()
    if len(cand) > 0:
        snr_to_sample[int(s)] = int(cand[0])

for snr_db in sorted(snr_to_sample.keys()):
    idx = snr_to_sample[snr_db]
    img_np = load_img_np(test_paths[idx])
    pred_cls, pred_prob = top_class_and_prob(most_interesting_model, img_np)

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    fig.suptitle(f'{most_interesting_model} | SNR={snr_db} dB | pred={CLASS_NAMES[pred_cls]} ({pred_prob:.3f})')

    axes[0].imshow(img_np)
    axes[0].set_title('Original')
    axes[0].axis('off')

    for j, method_name in enumerate(['gradcam', 'lime', 'shap'], start=1):
        attn, _, _ = METHODS[method_name](most_interesting_model, img_np, target_class=pred_cls)
        axes[j].imshow(overlay(img_np, attn))
        axes[j].set_title(method_name.upper())
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()